# 02 - Exploratory Factor Analysis
This notebook calculates the overall historical return, risk and other characteristics of DM factor-indices.

# 1. Load and check processed data (again)

In [ ]:
# Import libraries

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Display settings
pd.set_option("display.max_columns", 10)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")

In [ ]:
# Define the path to the data directory

current_directory = Path.cwd()

if current_directory.name == "notebooks":
    repository_root = current_directory.parent
else:
    repository_root = current_directory

processed_data_directory = repository_root / "data" / "processed"

print("Repository root:", repository_root)
print("Processed data directory:", processed_data_directory)

In [ ]:
# Set data directory path

monthly_levels_path = (processed_data_directory / "monthly_index_levels.csv")

monthly_returns_full_path = (processed_data_directory / "monthly_returns_full.csv")

monthly_returns_common_path = (processed_data_directory / "monthly_returns_common.csv")

In [ ]:
# Check if the files exist

print("Monthly levels file exists:", monthly_levels_path.exists())
print("Monthly returns full file exists:", monthly_returns_full_path.exists())
print("Monthly returns common file exists:", monthly_returns_common_path.exists())

In [ ]:
# Load the processed data

monthly_returns = pd.read_csv(monthly_returns_common_path, index_col="date", parse_dates=True)


In [ ]:
# Check shape

print("Shape:", monthly_returns.shape)
print("Columns:", monthly_returns.columns.tolist())
print("Index type", type(monthly_returns.index))

In [ ]:
# Check Timeframe 

print("First Observation:", monthly_returns.index.min())
print("Last Observations:", monthly_returns.index.max())
print("Number of months:", len(monthly_returns))
print("Number of series:", monthly_returns.shape[1])

In [ ]:
# Data validation

print("Missing values:", monthly_returns.isna().sum().sum())
print("Duplicate dates:", monthly_returns.index.duplicated().sum())
print("Returns <= -100%", (monthly_returns <= -1).sum().sum())
print("Non-numeric columns", monthly_returns.select_dtypes(exclude="number").columns.tolist())

In [ ]:
# More tests

assert monthly_returns.index.is_monotonic_increasing
assert not monthly_returns.index.duplicated().any()
assert not monthly_returns.isna().any().any()
assert (monthly_returns > -1).all().all()

# 2. Statistical analysis

In [ ]:
# Show structure

monthly_returns.info()

In [ ]:
# Overall range of returns

return_ranges = pd.DataFrame({"minimum": monthly_returns.min(), 
                               "maximum": monthly_returns.max(),
                               "mean": monthly_returns.mean()})

return_ranges

In [ ]:
# Monthly statistics
monthly_statistics = pd.DataFrame({"mean_monthly_return": monthly_returns.mean(),
                                   "monthly_volatility": monthly_returns.std(),
                                   "minimum_monthly_return": monthly_returns.min(),
                                   "maximum_monthly_return": monthly_returns.max(),
                                   "positive_month_share": (monthly_returns > 0).mean(),
                                   "skewness": monthly_returns.skew(),
                                   "excess_kurtosis": monthly_returns.kurt()
})

monthly_statistics

# Annualized geometric return

$$ r_{\mathrm{annualized}} = \bigg( \prod_{t=1}^{T} (1 + r_t) \bigg)^{12/T} - 1 $$

where $ r_t $ are the simple periodic returns

In [ ]:
# Define annualized geometric returns function
def annualized_geometric_returns(
        returns,
        periods_per_year = 12
): 
    returns = returns.dropna()
    number_of_periods = len(returns)
    total_growth = (1 + returns).prod()

    annualized_return = (total_growth**(periods_per_year / number_of_periods) - 1)

    return annualized_return

In [ ]:
# Test the function

market_annualized_return = (annualized_geometric_returns(monthly_returns["market"]))

market_annualized_return

# Annualized volatility

$$\sigma_{\mathrm{annualized}} = \sigma_{\mathrm{monthly}}\sqrt{12}$$

In [ ]:
# Annualized volatility

annualized_volatility = (monthly_returns.std()*np.sqrt(12))

annualized_volatility

In [ ]:
# Summary

summary_statistics = pd.DataFrame({
    "annualized_return": monthly_returns.apply(annualized_geometric_returns),
    "annualized_volatility": annualized_volatility,
    "mean_monthly_returns": monthly_returns.mean(),
    "minimum_monthly_return": monthly_returns.min(),
    "maximum_monthly_returns": monthly_returns.max(),
    "positive_month_share": (monthly_returns > 0).mean(),
    "skewness": monthly_returns.skew(),
    "excess_kurtosis": monthly_returns.kurt()
})

summary_statistics

# Wealth index

$$W_t = \prod_{i=1}^{t} (1 + r_i)$$

In [ ]:
# Wealth index
wealth_index = (1 + monthly_returns).cumprod()

wealth_index

In [ ]:
# Plot

fig, ax = plt.subplots(figsize=(11,6))

wealth_index.plot(ax=ax)
ax.set_title("Growth of one unit each")
ax.set_xlabel("Time")
ax.set_ylabel("Wealth Index")
ax.grid(alpha=0.3)
plt.tight_layout
plt.show()

In [ ]:
# Logartihmic plot

fig, ax = plt.subplots(figsize=(11,6))

wealth_index.plot(ax=ax, logy=True)
ax.set_title("Logarithmic growth of one unit each")
ax.set_xlabel("Time")
ax.set_ylabel("Wealth Index (Logarithmic)")
ax.grid(alpha=0.3)
plt.tight_layout
plt.plot()

# Relative performance of factor indices to the broad market 
$RW \hat{=}$ Relative Weatlth
for $i \in$ {Value, Momentum, Quality, Min_Volatility, Size_Proxy}

$$RW_{t}^{i} = \frac{W_{t}^{i}}{W_{t}^{\mathrm{Market}}}$$

In [ ]:
# Calculation
relative_wealth = wealth_index.div(wealth_index["market"], axis="index")

relative_wealth = relative_wealth.drop(columns="market")

relative_wealth

In [ ]:
# Plot
fig, ax = plt.subplots(figsize=(11,6))

relative_wealth.plot(ax=ax)
ax.axhline(y=1, linestyle="--")
ax.set_title("Cumulative Performance of each factor relative to the market")
ax.set_xlabel("Time")
ax.set_ylabel("Relative Wealth")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.plot()

# Correlation matrix

$$\rho_{i,j} = \frac{\mathrm{cov}(r_i , r_j)}{\sigma_i \sigma_j}$$

In [ ]:
# Calculate correlation matrix

correlation_matrix = monthly_returns.corr()

correlation_matrix

In [ ]:
# Heatmap

fig, ax = plt.subplots(figsize=(9,7))

image = ax.imshow(correlation_matrix, vmin=-1, vmax=1)

ax.set_xticks(range(len(correlation_matrix.columns)))

ax.set_xticklabels(correlation_matrix.columns, rotation=45, ha="right")

ax.set_yticks(range(len(correlation_matrix.index)))

ax.set_yticklabels(correlation_matrix.index)

for row in range(len(correlation_matrix.index)):
    for column in range(len(correlation_matrix.columns)):
        value = correlation_matrix.iloc[row, column]

        ax.text(column, row, f"{value:.2f}", ha="center", va="center")

fig.colorbar(image, ax=ax, label="Correlation")
ax.set_title("Correlation of monthly factor index returns")
plt.tight_layout()
plt.plot()

In [ ]:
# Avarage correlation of each factor

avarage_correlations = (correlation_matrix.apply(lambda column: column.drop(labels = column.name).mean()).sort_values)

avarage_correlations

# Drawdown analysis

$$D_t = \frac{W_t}{max_{\tau \leq t}(W_\tau)} - 1$$

we calculate the drawdowns $D_t$ realtive to the historic maximum before $W_\tau$ using the wealth index

In [ ]:
# Calculate maximum drawdown
running_maximum = wealth_index.cummax()

drawdowns = (wealth_index / running_maximum -1)

maximum_drawdowns = drawdowns.min()

# Add them to the summary of statistics

summary_statistics["maximum_drawdown"] = maximum_drawdowns

summary_statistics["maximum_drawdown"]

# Calmar Ratio

$$\mathrm{Calmar} = \frac{r_{\mathrm{annualized}}}{|D_{\mathrm{max}}|}$$

In [ ]:
# Calculate Calmar Ratio

calmar_ratio = (summary_statistics["annualized_return"] / summary_statistics["maximum_drawdown"].abs())

# Add them to the summary of statistics
summary_statistics["calmar_ratio"] = calmar_ratio

summary_statistics["calmar_ratio"]